# पाठ ११ - एजेन्ट-टु-एजेन्ट (A2A) प्रोटोकल


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A प्रोटोकल के हो?

**एजेन्ट-टु-एजेन्ट (A2A) प्रोटोकल** एउटा खुला मानक हो जसले AI एजेन्टहरूलाई सञ्चार गर्न, एक आपसमा पत्ता लगाउन, र सहकार्य गर्न सक्षम बनाउँछ — भिन्न फ्रेमवर्कहरूमा बनाइएका वा फरक सेवाहरूमा होस्ट गरिएका अवस्थामा पनि।

Key concepts:

- **खोज** – एजेन्टहरूले आफ्नो क्षमताहरू वर्णन गर्ने *एजेन्ट कार्ड* प्रकाशित गर्छन्, जसले अन्य एजेन्टहरू (वा ऑर्केस्ट्रेटरहरू) लाई कामका लागि सही विशेषज्ञ फेला पार्न सजिलो बनाउँछ।
- **सन्देश आदानप्रदान** – एजेन्टहरूले साझा प्रोटोकल मार्फत संरचित सन्देशहरू आदानप्रदान गर्छन्, ताकि एउटा एजेन्टको अनुरोध अर्कोले यसको आन्तरिक कार्यान्वयन जे भए पनि बुझेर पूरा गर्न सकोस्।
- **कार्य जीवनचक्र** – A2Aले *submitted*, *working*, *completed*, र *failed* जस्ता अवस्थाहरू परिभाषित गर्छ, जसले ऑर्केस्ट्रेटरलाई कसरी जिम्मा दिइएको कार्य अगाडि बढिरहेको छ भन्ने बारे पूर्ण दृश्य प्रदान गर्छ।

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents into a workflow where each agent contributes its expertise and passes results to the next.


## विशेषीकृत यात्रा एजेन्टहरू सिर्जना गर्दै


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## वर्कफ्लो मार्फत बहु-एजेन्ट सहकार्य

हामी तीन एजेन्टहरूलाई अनुक्रमिक वर्कफ्लोमा जडान गर्छौं जुन A2A सन्देश आदान-प्रदानलाई प्रतिबिम्बित गर्छ:

1. **CurrencyExchangeAgent** प्रयोगकर्ताको अनुरोध प्राप्त गर्छ र मुद्रा सम्बन्धी मार्गदर्शन प्रदान गर्छ।
2. **ActivityPlannerAgent** समृद्ध सन्दर्भ प्राप्त गर्छ र गतिविधि सिफारिसहरू थप्छ।
3. **TravelManagerAgent** दुवै इनपुटहरूलाई संयोजन गरी अन्तिम यात्रा सारांश तयार गर्छ।


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## उत्पादन वातावरणमा A2A को समझ

उत्पादन वातावरणमा A2A प्रोटोकलले शक्तिशाली क्रस-सर्भिस परिदृश्यहरू खोल्छ:

| क्षमता | विवरण |
|---|---|
| **फ्रेमवर्क-बीच अन्तरक्रिया** | एउटा फ्रेमवर्कमा बनाइएको एजेन्टले कुनै पनि अन्य A2A-compliant फ्रेमवर्कमा बनाइएको एजेन्टलाई कार्य सुम्पन गर्न सक्छ, जसले वास्तविक संगठनहरूबीचको अन्तरसञ्चालन सक्षम बनाउँछ। |
| **सेवा सीमाहरू** | एजेन्टहरू अलग माइक्रोसर्भिसहरू, क्लाउड क्षेत्रहरू, वा फरक संगठनहरूमा पनि रहन सक्छन् भने पनि निर्बाध रूपमा सहयोग गर्न सक्छन्। |
| **गतिशील खोज** | एक समन्वयकले रनटाइममा Agent Card रजिस्ट्रीलाई सोधेर दिइएको उप-कार्यका लागि सबैभन्दा उपयुक्त विशेषज्ञ फेला पार्न सक्छ। |
| **Streaming & पुश सूचनाहरू** | A2A ले Server-Sent Events (SSE) मार्फत वास्तविक-समय प्रगति अपडेटहरू र लामो समयसम्म चल्ने कार्यहरूका लागि पुश सूचनाहरू समर्थन गर्छ। |

हामीले माथि बनाएको कार्यप्रवाह यो ढाँचाको एक साधारण, इन-प्रोसेस संस्करण हो। वास्तविक परिनियोजनमा प्रत्येक एजेन्टले एक HTTP endpoint एक्स्पोज गर्नेछ, एक Agent Card प्रकाशित गर्नेछ, र A2A JSON-RPC प्रोटोकल मार्फत संवाद गर्नेछ।


## सारांश

यस पाठमा तपाईंले सिक्नुभयो:

1. **A2A प्रोटोकल के हो** — एजेन्ट-देखि-एजेन्ट पत्ता लगाउने, सन्देश आदानप्रदान गर्ने,
   र कार्य व्यवस्थापनको लागि खुला मापदण्ड।
2. **विशेषीकृत एजेन्टहरू कसरी सिर्जना गर्ने** — एक करेन्सी एक्सचेञ्ज एजेन्ट, एक गतिविधि योजनाकार एजेन्ट,
   र एक ट्राभल म्यानेजर समन्वयक।
3. **एजेन्टहरूलाई कार्यप्रवाहमा कसरी जडान गर्ने** — `WorkflowBuilder` प्रयोग गरेर क्रमिक
   सन्देश आदानप्रदान मोडल बनाउने।
4. **A2A उत्पादनमा कसरी काम गर्छ** — विभिन्न फ्रेमवर्क र सेवाहरूबीच सहयोग सक्षम पार्दै,
   डायनामिक पत्ता लगाउने र स्ट्रिमिङ अपडेटहरूसहित।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
यो दस्तावेज AI अनुवाद सेवा Co-op Translator (https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी शुद्धता सुनिश्चित गर्न प्रयासरत छौं, तर कृपया जानकार हुनुहोस् कि स्वचालित अनुवादमा त्रुटि वा अशुद्धता हुनसक्छ। मूल दस्तावेजलाई यसको मूल भाषामा नै आधिकारिक स्रोत मानिनु पर्छ। महत्वपूर्ण जानकारीका लागि पेशेवर मानवीय अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न हुने कुनै पनि भ्रम वा गलत व्याख्याका लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
